# 05 · Closed-set classical baselines

## Purpose
How far do the 144 physics features (R1) go on their own, with standard classifiers and no tuning? Rungs A0 (chance
references) and A1 (LightGBM, random forest, SVM-RBF, k-NN, logistic regression and an equal-weight soft-vote ensemble),
trained on `split_closed` **train** windows and evaluated on **validation** recordings. The test split stays sealed:
nothing here reads test rows, predictions or metrics. A primary-vs-capped sensitivity check, an envelope diagnostic
and permutation importance follow; the leaky-vs-honest illustration is a separate section at the end.

## Inputs
`results/baselines/*.json|csv` and `results/scores/*.parquet`, written by `python scripts/05_baselines.py closed`
(training and evaluation live in the script; this notebook only reads and plots). Fixed hyperparameters are in the
`baselines` section of `configs/base.yaml`.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the settings, <code>metadata.csv</code> and the saved validation results, and stops if those results were produced under different split assignments than the current ones.</div>

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml.cache import check_split_hash
from ml.config import load_config
from ml.labels import CLASS6, CLASS10
from ml.paths import DEFAULT_CONFIG, SYSTEM_CODE_DIR
from ml.splits import split_hash

cfg = load_config(DEFAULT_CONFIG)
meta = pd.read_csv(SYSTEM_CODE_DIR / "metadata.csv")
RES = SYSTEM_CODE_DIR / "results" / "baselines"
FIG = SYSTEM_CODE_DIR / "results" / "figures"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
closed = json.loads((RES / "closed_val.json").read_text())
prov = closed["provenance"]
check_split_hash(prov["split_hash"], meta, "closed_val.json")
print(f"feature table {prov['feature_table']} ({prov['n_features']} features); train {prov['n_train_recordings']} "
      f"recordings, validation {prov['n_val_recordings']}; git {prov['git_sha'][:12]}; split hash {split_hash(meta)[:12]}…")

## 1. Baseline table (validation, recording level)
Every model's window probabilities are turned into one prediction per recording the same way: clip to [1e-6, 1],
average the log-probabilities over the 9 windows, exponentiate, renormalise, take the argmax. class6 sums those
recording-level class10 probabilities. Macro-F1 carries a 95 % bootstrap interval over validation recordings;
window-level macro-F1 is shown for contrast.

**About the intervals.** Where every validation recording is classified correctly, the empirical bootstrap interval is [1.000, 1.000] and paired differences are [0, 0]. These are **degenerate**: resampling an all-correct set cannot produce errors it never contained. They are not evidence of zero uncertainty, nor of general equivalence between models.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Builds the main results table: for each model, its rung, recording-level macro-F1 with its 95 % interval for class10 and class6, balanced accuracy, window-level macro-F1, training time and prediction time per recording.</div>

In [ ]:
order = ["majority", "stratified", "lgbm", "rf", "svm", "knn", "logreg", "ensemble"]
rows = []
for name in order:
    r = closed["models"][name]
    c10, c6 = r["class10"], r["class6"]
    rows.append({"model": name, "rung": "A0" if name in ("majority", "stratified") else "A1",
                 "class10 macro-F1": f"{c10['macro_f1']:.3f} [{c10['macro_f1_ci'][0]:.3f}, {c10['macro_f1_ci'][1]:.3f}]",
                 "class10 balanced acc": round(c10["balanced_accuracy"], 3),
                 "class6 macro-F1": f"{c6['macro_f1']:.3f} [{c6['macro_f1_ci'][0]:.3f}, {c6['macro_f1_ci'][1]:.3f}]",
                 "class6 balanced acc": round(c6["balanced_accuracy"], 3),
                 "window macro-F1 (class10)": round(r["window_macro_f1"], 3),
                 "train s": round(r.get("train_seconds", np.nan), 1),
                 "ms per recording": round(r.get("ms_per_recording", np.nan), 2)})
table = pd.DataFrame(rows).set_index("model")
table.to_csv(RES / "closed_val_table.csv")
table

Per-class F1 (class10, validation recordings):

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows each model's F1 score for every one of the 10 classes on the validation recordings, so it is visible which classes each model gets right or wrong.</div>

In [ ]:
pd.DataFrame({m: closed["models"][m]["class10"]["per_class_f1"] for m in order}).loc[list(CLASS10)].round(3)

Confusion matrices for LightGBM and the ensemble (validation recordings; rows = true class):

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Draws the confusion matrices of LightGBM and the ensemble, for class10 and class6: each cell counts the validation recordings of the row's true class predicted as the column's class.</div>

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
for j, name in enumerate(["lgbm", "ensemble"]):
    for i, (view, labels) in enumerate([("class10", CLASS10), ("class6", CLASS6)]):
        cm = np.array(closed["models"][name][view]["confusion"])
        ax = axes[i, j]
        ax.imshow(cm, cmap="Blues")
        ax.set_xticks(range(len(labels)), labels, rotation=60, ha="right", fontsize=7)
        ax.set_yticks(range(len(labels)), labels, fontsize=7)
        for (a, b), v in np.ndenumerate(cm):
            if v:
                ax.text(b, a, v, ha="center", va="center", fontsize=7, color="white" if v > cm.max() / 2 else "black")
        ax.set(title=f"{name}, {view}", xlabel="predicted", ylabel="true")
        ax.grid(False)
fig.tight_layout()
fig.savefig(FIG / "05_confusion_val.png")

## 2. Sensitivity: primary vs capped at 10 kHz (LightGBM)
Same model and hyperparameters on the r1cap10 table (low-pass at 10 kHz before every feature). **Interpretation rule:**
this is a sensitivity experiment; a drop would show dependence on the information above 10 kHz, not prove that
information was an artefact. Reported only; the choice of pipeline is Bhuvesh's. If both reach the ceiling, that is an observation that the variant retained validation performance; it does not prove equivalent generalisation or resolve the artefact concern.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Compares LightGBM on the primary and the capped feature tables: per-class F1 side by side, and the paired bootstrap difference in recording-level macro-F1 (the same resampled validation recordings for both).</div>

In [ ]:
sens = closed["sensitivity"]
d = sens["paired_primary_minus_capped"]
print(f"capped table {sens['table']}")
print(f"macro-F1 class10: primary {sens['primary']['class10']['macro_f1']:.3f}, capped {sens['capped']['class10']['macro_f1']:.3f}; "
      f"primary − capped {d['diff']:+.3f}, 95 % CI [{d['ci'][0]:+.3f}, {d['ci'][1]:+.3f}], P(diff > 0) = {d['p_positive']:.2f}")
print(f"macro-F1 class6: primary {sens['primary']['class6']['macro_f1']:.3f}, capped {sens['capped']['class6']['macro_f1']:.3f}")
pd.DataFrame({"primary": sens["primary"]["class10"]["per_class_f1"], "capped at 10 kHz": sens["capped"]["class10"]["per_class_f1"]}
             ).loc[list(CLASS10)].assign(difference=lambda t: t["primary"] - t["capped at 10 kHz"]).round(3)

## 3. Diagnostic: LightGBM with vs without the envelope features
The 34 envelope features (1–5 kHz band) removed, everything else identical. Reported only; alternative bands are
compared later under the agreed protocol. At the ceiling, "no difference" means no measurable incremental benefit on this split, nothing more.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Compares LightGBM trained with and without the envelope features: per-class F1 side by side and the paired bootstrap difference in recording-level macro-F1.</div>

In [ ]:
env = closed["envelope_diagnostic"]
d = env["paired_with_minus_without"]
print(f"macro-F1 class10: with {closed['models']['lgbm']['class10']['macro_f1']:.3f}, without "
      f"{env['without_envelope']['class10']['macro_f1']:.3f} ({env['n_features_without']} features); with − without "
      f"{d['diff']:+.3f}, 95 % CI [{d['ci'][0]:+.3f}, {d['ci'][1]:+.3f}], P(diff > 0) = {d['p_positive']:.2f}")
pd.DataFrame({"with envelope": closed["models"]["lgbm"]["class10"]["per_class_f1"],
              "without envelope": env["without_envelope"]["class10"]["per_class_f1"]}
             ).loc[list(CLASS10)].assign(difference=lambda t: t["with envelope"] - t["without envelope"]).round(3)

## 4. Permutation importance (LightGBM, validation, recording level)
The drop in recording-level macro-F1 when a feature, or a whole channel family, is shuffled **between validation
recordings as whole 9-window blocks** (windows are never shuffled individually), averaged over 5 repeats.

Permutation importance measures **this model's reliance** on a feature or family; it says nothing about whether that evidence is physical or artefactual.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the saved importances and plots the 20 most important single features and every channel family, with the spread over the 5 shuffles as error bars.</div>

In [ ]:
fi = pd.read_csv(RES / "permutation_importance_features.csv")
fam = pd.read_csv(RES / "permutation_importance_families.csv")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
top = fi.head(20).iloc[::-1]
axes[0].barh(top.group, top.importance, xerr=top["std"], color="#4c72b0")
axes[0].set(xlabel="drop in recording-level macro-F1", title="Top 20 features")
axes[0].tick_params(axis="y", labelsize=7)
f2 = fam.iloc[::-1]
axes[1].barh(f2.group, f2.importance, xerr=f2["std"], color="#dd8452")
axes[1].set(xlabel="drop in recording-level macro-F1", title="Channel families (columns shuffled together)")
fig.tight_layout()
fig.savefig(FIG / "05_permutation_importance.png")
display(fi.head(20).round(4))
fam.round(4)

## 5. Why is the closed-set task this easy? (diagnostic)
Nearly every A1 model reaches recording-level macro-F1 1.000 on validation, so the sensitivity, envelope and
importance comparisons above have no room to show a difference. Two integrity checks, then the structure behind it:
for each validation recording, its nearest training recording in feature space (recording-mean features, standardised
with means and standard deviations fitted on the **training recordings only**).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Checks that no recording appears twice (identical checksums) and that train and validation share no recordings, then standardises the recording-average features with statistics from the training recordings only, finds each validation recording's nearest training recording and reports how often that neighbour has the same class, the same severity folder, and how far apart their speeds are.</div>

In [ ]:
from ml.cache import load_features
from ml.features import feature_columns
from ml.paths import data_dir

pre = cfg.preprocess
print(f"duplicate npy checksums: {int(meta.npy_sha256.duplicated(keep=False).sum())}; train ∩ val recordings: "
      f"{len(set(meta.rec_id[meta.split_closed == 'train']) & set(meta.rec_id[meta.split_closed == 'val']))}")
path = max((data_dir() / "features").glob(f"{prov['feature_table'].split('_')[0]}_*.parquet"), key=lambda q: q.stat().st_mtime)
f = load_features(path, meta, pre)
f = f[f.split_closed.isin(["train", "val"])]
fc = feature_columns(f)
rec = f.groupby(["rec_id", "split_closed", "class10", "severity", "fr_measured_hz"])[fc].mean().reset_index()
is_tr = (rec.split_closed == "train").to_numpy()
mu, sd = rec.loc[is_tr, fc].mean(), rec.loc[is_tr, fc].std()  # scaling fitted on training recordings only
Z = ((rec[fc] - mu) / sd).to_numpy()
A, B = Z[is_tr], Z[~is_tr]
dist = np.sqrt(((B[:, None, :] - A[None, :, :]) ** 2).sum(-1))
nn = rec[is_tr].iloc[dist.argmin(1)].reset_index(drop=True)
vv = rec[~is_tr].reset_index(drop=True)
pd.Series({"validation recordings": len(vv),
           "nearest train recording has the same class": (nn.class10.values == vv.class10.values).mean(),
           "... and the same severity folder": ((nn.class10.values == vv.class10.values) & (nn.severity.values == vv.severity.values)).mean(),
           "median speed difference to it (Hz)": np.median(np.abs(nn.fr_measured_hz.values - vv.fr_measured_hz.values))}).round(3)

No recording is duplicated and train and validation share none. With the scaling fitted on training recordings only,
95.6 % of validation recordings have a same-class nearest neighbour in training and 69.0 % one from the same class and
severity folder, typically the adjacent speed step (median 0.742 Hz away). Matching class and severity folders
**suggests the closed-set task is close to interpolation** between neighbouring recordings; it does not by itself
establish that they come from the same acquisition session (folders are only a proxy for sessions). Generalisation
beyond this split is tested by the protocols that hold out whole regions (`split_severity`, `split_speed_block`, LOCO)
and by the folder-grouped development evaluation (notebook 05b).

## Findings
- **Several classical models achieve perfect classification on the current closed-set validation split. Generalisation beyond that split remains untested.** Recording-level class10 macro-F1 on validation (294 recordings): 1.000 for LightGBM, random forest, SVM, logistic regression and the equal-weight ensemble, 0.996 [0.988, 1.000] for k-NN; class6 1.000 for all. Chance references: majority 0.029, stratified 0.100 [0.060, 0.139]. Window-level macro-F1 0.994–1.000.
- **The [1.000, 1.000] intervals and the paired [0, 0] differences are degenerate empirical bootstrap intervals**: resampling an all-correct set cannot produce unseen errors. They are not evidence of zero uncertainty or of general equivalence.
- **Nearest-neighbour diagnostic** (scaling fitted on training recordings only): 95.6 % same class, 69.0 % same class and severity folder, median 0.742 Hz apart. This suggests interpolation between neighbouring recordings; it does not by itself establish acquisition-session identity.
- **Capped at 10 kHz:** the variant retained validation performance (1.000). That is an observation; it does not prove equivalent generalisation or resolve the artefact concern. **Envelope features:** no measurable incremental benefit on this split.
- **Permutation importance** (whole 9-window blocks): by family, microphone 0.206, over_axial 0.201, over_radial 0.109, under_axial 0.086; the envelope families 0. This measures the model's reliance on each family, not whether the evidence is physical or artefactual.
- **Model selection** uses permitted development data only; closed-set ties are broken by simplicity and measured latency; reserved tests are never used for selection.

## Outputs written
- `results/baselines/closed_val.json`, `closed_val_table.csv`, `permutation_importance_features.csv`, `permutation_importance_families.csv`
- `results/scores/closed_r1_<model>.parquet` (train/val windows only; gitignored, reproducible with the script)
- `results/figures/05_confusion_val.png`, `results/figures/05_permutation_importance.png`
- MLflow runs in experiment `closed_baselines` (`sqlite:///mlruns.db`)
- Separate section below: `results/baselines/leaky_vs_honest.json`, `results/figures/05_leaky_vs_honest.png`

---
# SEPARATE: deliberate leakage illustration
**Not a valid result.** This section shows, on purpose, what a window-level split does. It is reported apart from the
baselines above and must not be quoted as model performance.

LightGBM (same fixed hyperparameters) on train + val windows only, 5 folds each:
- **(a) leaky:** a random 5-fold split at **window** level, so windows of one recording land on both sides;
- **(b) honest:** the recording-grouped `cv_fold`.

LightGBM needs no scaler, so there is no other learned preprocessing to refit inside the folds. The signal-cleaning
parameters (burst threshold, band-stop, envelope band) are fixed and reused, so both numbers are **conditional on the
fixed preprocessing**; this is not a fully nested evaluation. Written by `python scripts/05_baselines.py leaky`.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the leakage illustration's results and plots window-level macro-F1 for each of the 5 folds under the window-level split and the recording-grouped split, with the share of test windows whose recording also had windows in training.</div>

In [ ]:
leak = json.loads((RES / "leaky_vs_honest.json").read_text())
check_split_hash(leak["provenance"]["split_hash"], meta, "leaky_vs_honest.json")
names = {"window_random": "(a) random split at window level", "recording_grouped": "(b) recording-grouped cv_fold"}
fig, ax = plt.subplots(figsize=(8, 3.8))
for k, (key, label) in enumerate(names.items()):
    s = leak[key]["fold_macro_f1"]
    ax.scatter(np.full(len(s), k) + np.linspace(-0.08, 0.08, len(s)), s, color=f"C{k}", zorder=3)
    ax.hlines(np.mean(s), k - 0.25, k + 0.25, color=f"C{k}", lw=2)
ax.set_xticks(range(2), [f"{l}\ntest windows with their recording in training: "
                         f"{leak[k]['share_of_test_windows_with_their_recording_in_training']:.0%}" for k, l in names.items()],
              fontsize=8)
ax.set(ylabel="window-level macro-F1 (5 folds)", title="Deliberate leakage illustration (not a valid result)")
fig.tight_layout()
fig.savefig(FIG / "05_leaky_vs_honest.png")
pd.DataFrame({names[k]: {"mean": leak[k]["mean"], "std": leak[k]["std"],
                         "share of test windows with their recording in training": leak[k]["share_of_test_windows_with_their_recording_in_training"]}
              for k in names}).T.round(4).assign(gap_vs_honest=lambda t: t["mean"] - leak["recording_grouped"]["mean"])

**Illustration result.** The window-level split scores 1.000 ± 0.000: every test window's recording also had windows
in training. The recording-grouped split scores 0.991 ± 0.005. The gap (+0.009) is small **because the closed-set task
is near-saturated even when split honestly** (section 5): the leak removes the last errors rather than inflating a
modest score. The illustration shows the mechanism; its size here says more about how easy the closed task is than about
how large window leakage can be on a harder protocol.